# 02 — Querying the REF API

This notebook shows how to talk to the public REF API: set up the client, discover
diagnostics, retrieve metric values, and inspect an execution and its output files.

**Prerequisites:** [01 — REF concepts](01-ref-concepts.ipynb).

**What you need:** an internet connection.

## The API and its client

The REF API is documented with an [OpenAPI](https://www.openapis.org) schema. You can
browse the interactive docs at <https://api.climate-ref.org/docs> — every query used
below is described there.

The `climate_ref_client` package is *generated* from that schema (see
`scripts/generate_client.sh`). The `ref_tutorials` helper builds a client for us:

In [ ]:
from ref_tutorials import get_client

client = get_client()

Each API endpoint is a function under `climate_ref_client.api`. The functions take a
`client=` argument and have a `.sync(...)` method that performs the request and returns
a parsed response. The payload is on the `.data` attribute.

## Listing diagnostics

In [ ]:
from climate_ref_client.api.diagnostics import diagnostics_list

diagnostics = diagnostics_list.sync(client=client).data
len(diagnostics)

In [ ]:
import pandas as pd

pd.DataFrame(
    {"provider": d.provider.slug, "slug": d.slug, "name": d.name}
    for d in sorted(diagnostics, key=lambda d: (d.provider.slug, d.slug))
)

## Retrieving metric values

Many diagnostics expose **scalar** metric values. We pick one diagnostic and ask the
API for its scalar values. Each value carries a set of *dimensions* (model, statistic,
region, ...) and a single number.

In [ ]:
from climate_ref_client.api.diagnostics import diagnostics_list_metric_values
from climate_ref_client.models.metric_value_type import MetricValueType

diagnostic = next(d for d in diagnostics if d.execution_groups)

values = diagnostics_list_metric_values.sync(
    diagnostic.provider.slug,
    diagnostic.slug,
    value_type=MetricValueType.SCALAR,
    client=client,
).data
print(f"{diagnostic.name}: {len(values)} scalar metric values")
values[0]

Raw metric values are awkward to work with. The `ref_tutorials` helper flattens them
into a tidy `pandas.DataFrame` — one column per dimension, plus a `value` column:

In [ ]:
from ref_tutorials import metric_values_to_dataframe

df = metric_values_to_dataframe(values)
df.head(10)

From here it is ordinary pandas — filter, group, and summarise as you like:

In [ ]:
df["value"].describe()

## Inspecting an execution

Beyond metric values, executions produce **output files** (NetCDF data, figures). We
fetch one execution group and list what it produced.

In [ ]:
from climate_ref_client.api.executions import executions_get

execution = executions_get.sync(diagnostic.execution_groups[0], client=client)
print("execution key:", execution.key)

outputs = execution.latest_execution.outputs
for output in outputs[:10]:
    print("  ", output.filename)

Each output has a `url`. You can download it with `requests` and open NetCDF files with
`xarray` for your own analysis — that is exactly what notebook 03 does to build a figure.

## Recap

- Build a client with `get_client()`.
- API endpoints live under `climate_ref_client.api`; call `.sync(client=client)` and
  read `.data`.
- `diagnostics_list` discovers diagnostics; `diagnostics_list_metric_values` fetches
  their numbers; `executions_get` inspects a single run.
- `metric_values_to_dataframe` turns metric values into tidy pandas.

**Next:** [03 — A publication-ready figure](03-publication-figure.ipynb).